Pastas a copiar para o dataset completo com pacientes válidos

In [ ]:
import os
import pandas as pd
import shutil
from tqdm import tqdm
import glob

# --- CONFIGURAÇÃO ---
PASTAS_RAIZ = [
    # "/media/filipe/Dados/MIMIC-4",
    "/media/filipe/Datasets/Datasets/MIMIC-3/MIMIC-3_pacientes_novos_18_25",
    "/media/filipe/Dados/MIMIC-3/MIMIC-3_pacientes_novos_26_39",
    "/media/filipe/Datasets/Datasets/MIMIC-3/MIMIC-3_pacientes_novos_40_50",
    "/media/filipe/Dados/MIMIC-3/MIMIC-3_pacientes_novos_51_55",
    "/media/filipe/Datasets/Datasets/MIMIC-3/MIMIC-3_pacientes_novos_60_65",
]

# Pasta onde ficam centralizadas as pastas dos pacientes
PASTA_DESTINO_FINAL = "/media/filipe/Datasets/Datasets/MIMIC_DATASET_CONSOLIDADO"

def consolidar_por_csv_index():
    if not os.path.exists(PASTA_DESTINO_FINAL):
        os.makedirs(PASTA_DESTINO_FINAL)

    pacientes_copiados = set()

    for pasta_base in PASTAS_RAIZ:
        if not os.path.exists(pasta_base):
            print(f"Pasta raiz não encontrada: {pasta_base}")
            continue

        # CSVs de indexação (MIMIC-3 e MIMIC-4)
        # O padrão busca ficheiros que comecem por 'dataset_final_para_treino'
        ficheiros_index = glob.glob(os.path.join(pasta_base, "dataset_final_para_treino_*.csv"))

        for csv_path in ficheiros_index:
            print(f"\nLendo index: {os.path.basename(csv_path)}")
            try:
                df = pd.read_csv(csv_path)
                
                # Verificação de colunas críticas
                if 'absolute_path' not in df.columns or 'subject_folder' not in df.columns:
                    print(f"  Colunas necessárias não encontradas em {os.path.basename(csv_path)}")
                    continue

                # Itera pelos registros do CSV
                for _, row in tqdm(df.iterrows(), total=len(df), desc="Pacientes"):
                    sub_id = str(row['subject_folder'])
                    abs_path = row['absolute_path']
                    
                    # Se este paciente já foi consolidado, ignoramos
                    if sub_id in pacientes_copiados:
                        continue

                    # O absolute_path aponta para o ficheiro (ex: .../p000188/3317157_0002)
                    # O dirname dá-nos a pasta do paciente (ex: .../p000188)
                    pasta_origem_paciente = os.path.dirname(abs_path)
                    pasta_destino_paciente = os.path.join(PASTA_DESTINO_FINAL, sub_id)

                    if os.path.exists(pasta_origem_paciente):
                        if not os.path.exists(pasta_destino_paciente):
                            try:
                                # Copia a pasta inteira do paciente com todos os sinais
                                os.system(f"rsync -ah --progress '{pasta_origem_paciente}/' '{pasta_destino_paciente}'")
                                pacientes_copiados.add(sub_id)
                            except Exception as e:
                                print(f"\n  Erro ao copiar {sub_id}: {e}")
                        else:
                            # Se já existe no destino (talvez de outra pasta raiz), apenas registamos
                            pacientes_copiados.add(sub_id)
                    else:
                        # Caso o caminho no CSV esteja quebrado ou o disco não esteja montado
                        pass 

            except Exception as e:
                print(f"  Erro ao processar CSV {csv_path}: {e}")

    print("\n" + "="*45)
    print("CONSOLIDAÇÃO CONCLUÍDA COM SUCESSO")
    print(f"Destino: {PASTA_DESTINO_FINAL}")
    print(f"Total de pacientes únicos movidos: {len(pacientes_copiados)}")
    print("="*45)

if __name__ == "__main__":
    consolidar_por_csv_index()

Gerar csv de features

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
# Pasta com os sinais consolidados
PASTA_CONSOLIDADA = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"

# Caminhos dos metadados originais
MIMIC3_PAT = "/home/filipe/Transferências/mimic-iii-1.4/PATIENTS.csv"
MIMIC3_ADM = "/home/filipe/Transferências/mimic-iii-1.4/ADMISSIONS.csv"
MIMIC4_PAT = "/home/filipe/Transferências/mimic-iv-3.1/hosp/patients.csv"

OUTPUT_MASTER_CSV = "/media/filipe/Dados/features_todos_consolidado.csv"

def carregar_metadados_totais():
    print("A carregar e processar metadados MIMIC-III e IV...")
    
    # --- PROCESSAR MIMIC-III ---
    df3_p = pd.read_csv(MIMIC3_PAT)
    df3_a = pd.read_csv(MIMIC3_ADM)
    df3_p["DOB"] = pd.to_datetime(df3_p["DOB"], errors="coerce")
    df3_a["ADMITTIME"] = pd.to_datetime(df3_a["ADMITTIME"], errors="coerce")
    
    df3_merged = df3_a.merge(df3_p[['SUBJECT_ID', 'DOB', 'GENDER']], on="SUBJECT_ID")
    df3_merged['age'] = df3_merged['ADMITTIME'].dt.year - df3_merged['DOB'].dt.year
    df3_merged.loc[df3_merged['age'] > 89, 'age'] = 90
    
    # Limpeza de IDs para o III (remover zeros à esquerda)
    df3_meta = df3_merged.groupby('SUBJECT_ID').agg({'age': 'first', 'GENDER': 'first'}).reset_index()
    df3_meta['subject_id_clean'] = df3_meta['SUBJECT_ID'].astype(str)
    df3_meta = df3_meta.rename(columns={'GENDER': 'gender'})[['subject_id_clean', 'age', 'gender']]

    # --- PROCESSAR MIMIC-IV ---
    df4_p = pd.read_csv(MIMIC4_PAT)
    df4_meta = df4_p[['subject_id', 'anchor_age', 'gender']].copy()
    df4_meta = df4_meta.rename(columns={'subject_id': 'subject_id_clean', 'anchor_age': 'age'})
    df4_meta['subject_id_clean'] = df4_meta['subject_id_clean'].astype(str)

    # Combinar ambos num dicionário para busca ultra-rápida {id: (age, gender)}
    full_meta = pd.concat([df3_meta, df4_meta]).set_index('subject_id_clean')
    return full_meta.to_dict('index')

def gerar_master_csv():
    meta_dict = carregar_metadados_totais()
    
    print(f"Mapeando ficheiros em: {PASTA_CONSOLIDADA}")
    records = []
    
    # Listar pastas de pacientes
    pastas_pacientes = [d for d in os.listdir(PASTA_CONSOLIDADA) if os.path.isdir(os.path.join(PASTA_CONSOLIDADA, d))]

    for s_folder in tqdm(pastas_pacientes, desc="Pacientes"):
        # Lógica de ID dos ficheiros originais
        # MIMIC-III: p000543 -> 543 | MIMIC-IV: p10014354 -> 10014354
        clean_id = s_folder.replace('p', '').lstrip('0')
        
        info = meta_dict.get(clean_id, {'age': None, 'gender': None})
        
        caminho_paciente = os.path.join(PASTA_CONSOLIDADA, s_folder)
        ficheiros_hea = [f for f in os.listdir(caminho_paciente) if f.endswith('.hea') and '_' in f]

        for hea in ficheiros_hea:
            record_name = hea.replace('.hea', '')
            abs_path = os.path.join(caminho_paciente, record_name)
            
            records.append({
                'subject_folder': s_folder,
                'segment': record_name,
                'absolute_path': abs_path,
                'age': info['age'],
                'gender': info['gender']
            })

    df_final = pd.DataFrame(records)
    
    # Remover entradas sem metadados (se houver algum erro de ID)
    df_final = df_final.dropna(subset=['age', 'gender'])
    
    df_final.to_csv(OUTPUT_MASTER_CSV, index=False)
    print(f"\nSUCESSO! Master CSV criado com {len(df_final)} segmentos.")
    print(f"Colunas: {list(df_final.columns)}")
    print(f"Localização: {OUTPUT_MASTER_CSV}")

if __name__ == "__main__":
    gerar_master_csv()